# SO-101 Perception Benchmark — SAM 3 + metric-depth vs known ground truth

Runs the **`find_object` perception components** on simulation frames whose object
centre is known **exactly**, and scores each against ground truth so we pick the
pipeline on measured error, not vibes.

**Two things measured, separately:**
1. **Detection (SAM 3)** — open-vocab text prompt (`"red cube"`) → predicted centre pixel.
   Scored vs the exact projected pixel (pixel error + detection rate).
2. **Metric depth** (Depth Pro / UniDepthV2 / Metric3D-v2) — depth at a pixel → back-project
   with `camera_math.point_at_depth` → world (x,y,z). Scored vs the exact world centre, both
   at the **GT pixel** (isolates depth) and at the **SAM pixel** (full pipeline).

## Honesty
Ground truth is used **only** to (a) know which pixel is the true centre and (b) score error.
No GT drives any estimate. Every depth backend must return depth in **metres along the camera
OPTICAL / FORWARD axis** — the convention `camera_math.point_at_depth` consumes.

## Workflow
Clone this repo on Colab and run top-to-bottom. Depth backends have conflicting deps — run
**one at a time** (install cell per backend). When something breaks, tell the author which
cell + error; they push a fix and you `!git pull`.

## 0. Setup — clone/cd + imports

In [ ]:
# On Colab: clone the repo (set CLONE_URL after first push) and cd into it.
import os, sys
CLONE_URL = 'https://github.com/Yunsmn/RoboticsPerceptionTest.git'
if not os.path.exists('camera_math.py'):
    repo = CLONE_URL.rstrip('/').split('/')[-1].removesuffix('.git')
    if not os.path.exists(repo):
        os.system(f'git clone {CLONE_URL}')
    os.chdir(repo)
sys.path.insert(0, '.')
import numpy as np, json, math
from pathlib import Path
import camera_math as CM      # pure-numpy; standalone copy of the sim's camera model
print('camera_math OK:', np.round(CM.side_cam_params()[:3], 2))

## 0b. Run logger — capture EVERYTHING to `run_log.md`
Run this once, right after setup. It tees every cell's source + output + full tracebacks to
`run_log.md`. At the end call `download_log()` to pull the file to your machine and send it to
the author — no copy-paste, nothing truncated.

In [ ]:
# Logs every subsequent cell (source + stdout/stderr + full tracebacks) to run_log.md.
import sys, io, datetime, traceback, subprocess
from IPython import get_ipython
_LOG = 'run_log.md'; _f = open(_LOG, 'w')
def _w(s=''):
    _f.write(str(s) + '\n'); _f.flush()
_w('# Run log — ' + datetime.datetime.now().isoformat(timespec='seconds'))
try:
    _gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
except Exception:
    _gpu = ''
_w('- GPU: ' + (_gpu or 'none / CPU')); _w('- Python: ' + sys.version.split()[0])
class _Tee:
    # Wrap the LIVE notebook stream (ipykernel) so prints STILL SHOW and also get logged.
    _is_tee = True
    def __init__(self, real): self._real = real
    def write(self, s):
        n = self._real.write(s)
        try: _f.write(s); _f.flush()
        except Exception: pass
        return n
    def flush(self):
        try: self._real.flush()
        except Exception: pass
    def __getattr__(self, k):
        return getattr(self.__dict__['_real'], k)
if not getattr(sys.stdout, '_is_tee', False): sys.stdout = _Tee(sys.stdout)
if not getattr(sys.stderr, '_is_tee', False): sys.stderr = _Tee(sys.stderr)
_ip = get_ipython(); _n = {'i': 0}
def _pre(info):
    _n['i'] += 1
    _w(); _w('## Cell ' + str(_n['i'])); _w('```python')
    _w((info.raw_cell or '').rstrip()); _w('```'); _w('**output:**'); _w('```text')
def _post(res):
    _w('```')
    err = getattr(res,'error_in_exec',None) or getattr(res,'error_before_exec',None)
    if err is not None:
        _w('**ERROR:**'); _w('```text')
        _w(''.join(traceback.format_exception(type(err), err, err.__traceback__))); _w('```')
if not globals().get('_LOGGER_ON'):
    _ip.events.register('pre_run_cell', _pre); _ip.events.register('post_run_cell', _post)
    _LOGGER_ON = True
def download_log():
    """Call when done -> pulls run_log.md to your machine (or prints its path)."""
    _f.flush()
    try:
        from google.colab import files; files.download(_LOG)
    except Exception as e:
        print('download failed (%s) - file is at %s' % (e, _LOG))
print('run logger active -> run_log.md   (call download_log() at the end)')

## 1. Install a depth model (the main result)
Uncomment **one** metric-depth model, run it (**takes minutes**), then **Runtime → Restart**, then
run §2–§3 and §6. Their deps conflict, so one per session. SAM 3 detection is separate and gated
(§4). A plain run finishing in 0s is expected — nothing installs until you pick a model.

In [ ]:
# Uncomment ONE metric-depth model, run, then RESTART the runtime (numpy ABI), then run §6.
# !pip -q install git+https://github.com/apple/ml-depth-pro.git   # Depth Pro
# !pip -q install unidepth                                         # UniDepthV2
# !pip -q install mmcv mmengine timm                               # Metric3D-v2 (torch.hub)

### ⚠️ Restart the runtime after the install cell
`pip install` can leave the running kernel with a mismatched numpy — you'll see
*`ValueError: numpy.dtype size changed`* when a model imports. It's a Colab quirk, not a bug here.
Fix: after the install finishes → **Runtime → Restart session**, then run the logger cell and
continue. **Do NOT re-run the install cell** (already installed; re-running re-triggers it).

## 2. Data — sim frames with an EXACTLY known centre

`data/manifest.json` (from the sim harness) + `data/frames/pose_XXX.png`. Each frame carries
`gt_xyz` (world metres, arm-base frame), `gt_uv` (the exact projected centre pixel, self-checked
to 1e-16 m) and `gt_depth_m`. The camera intrinsics/extrinsics are in `manifest['camera']`.

In [ ]:
man = json.loads(Path('data/manifest.json').read_text())
cam = man['camera']
f, cx, cy = cam['f'], cam['cx'], cam['cy']
cam_pos = np.array(cam['cam_pos'], float)
R_cw    = np.array(cam['R_cw'], float)
W, H = cam['width'], cam['height']
FRAMES = man['frames']
print(len(FRAMES), 'frames |', f'{W}x{H}', '| f=%.1f' % f)

# HERO frame for the single-image demo (nearest the workspace centre).
hero = min(FRAMES, key=lambda fr: abs(fr['gt_xyz'][1]))
print('hero:', hero['png'], '| exact centre world', np.round(hero['gt_xyz'],4),
      '| exact centre pixel', hero['gt_uv'])

## 3. Camera model — project the known centre to its pixel
Inverse of `point_at_depth`: world → (u, v, forward-axis depth). Same convention the harness used.

In [ ]:
def project(P):
    '''World point -> (u, v, forward_axis_depth_m).'''
    xc, yc, zc = R_cw.T @ (np.asarray(P, float) - cam_pos)
    depth = -zc
    return cx + f * xc / depth, cy - f * yc / depth, depth

# sanity: our own projection must reproduce the manifest's stored gt_uv
u,v,_ = project(hero['gt_xyz'])
print('reproj vs manifest gt_uv:', np.round([u,v],3), 'vs', hero['gt_uv'])

## 4. SAM 3 detection (OPTIONAL — gated weights)
SAM 3 weights are **gated**. Skip this whole section if you just want the depth numbers (§6 scores
depth at the *known* pixel and needs no detector). To enable detection:
1. Request access at **https://huggingface.co/facebook/sam3** (accept the terms; wait for approval).
2. Add your HF token to Colab **🔑 Secrets** as `HF_TOKEN`.
3. Run the setup cell below (installs ultralytics + the CLIP fix, downloads `sam3.pt`).

In [ ]:
# OPTIONAL SAM 3 setup — needs approved access to facebook/sam3 + HF_TOKEN secret.
!pip -q install ultralytics
!pip -q install -U git+https://github.com/ultralytics/CLIP.git   # fixes SAM3 'SimpleTokenizer' error
from google.colab import userdata
from huggingface_hub import hf_hub_download
_w = hf_hub_download('facebook/sam3', 'sam3.pt', token=userdata.get('HF_TOKEN'), local_dir='.')
print('sam3.pt ->', _w)

In [ ]:
_pred = None
def sam3_center(img_rgb, prompt='red cube'):
    '''(u,v) of the best SAM 3 instance matching the TEXT prompt, or None. Needs gated sam3.pt.'''
    global _pred
    from ultralytics.models.sam import SAM3SemanticPredictor
    if _pred is None:
        _pred = SAM3SemanticPredictor(overrides=dict(conf=0.25, task='segment',
                                      mode='predict', model='sam3.pt', save=False, verbose=False))
    _pred.set_image(img_rgb)
    r = _pred(text=[prompt])[0]              # SAM 3 concept prompt uses text=[...] (not texts=)
    if getattr(r, 'masks', None) is None or len(r.masks) == 0:
        return None
    m = r.masks.data.cpu().numpy()
    conf = r.boxes.conf.cpu().numpy() if getattr(r, 'boxes', None) is not None else np.ones(len(m))
    best = m[int(np.argmax(conf))] > 0.5
    ys, xs = np.where(best)
    return (float(xs.mean()), float(ys.mean())) if len(xs) else None

### 4a. SAM 3 detection error over all frames

In [ ]:
from PIL import Image
def eval_sam(prompt='red cube'):
    px_err, miss = [], 0
    for fr in FRAMES:
        img = np.array(Image.open('data/'+fr['png']).convert('RGB'))
        c = sam3_center(img, prompt)
        if c is None:
            miss += 1; continue
        gu, gv = fr['gt_uv']
        px_err.append(math.hypot(c[0]-gu, c[1]-gv))
    e = np.array(px_err) if px_err else np.array([np.nan])
    return {'n': len(FRAMES), 'detected': len(px_err), 'miss': miss,
            'px_err_mean': float(np.nanmean(e)), 'px_err_median': float(np.nanmedian(e))}
# Runs only if gated sam3.pt is present; otherwise reports + skips so §6 (depth) still runs.
import os as _os
if _os.path.exists('sam3.pt'):
    print('SAM 3 detection over', len(FRAMES), 'frames...'); print(eval_sam())
else:
    print('SAM 3 weights (sam3.pt) not present — gated model, and OPTIONAL.')
    print('Do the SAM 3 setup cell to enable detection, or skip it — §6 depth needs no detector.')

## 5. Metric-depth adapters
Each returns an `(H, W)` float32 map of **forward-axis metric depth (metres)**. Convert range /
disparity / z as needed. Best-effort against each model's API — **the cells to iterate on**.

In [ ]:
def infer_depth_pro(img_rgb):
    import depth_pro, torch
    model, tf = depth_pro.create_model_and_transforms(); model.eval()
    with torch.no_grad():
        pred = model.infer(tf(img_rgb), f_px=f)
    return pred['depth'].squeeze().cpu().numpy().astype('float32')   # metric metres

def infer_unidepth_v2(img_rgb):
    import torch; from unidepth.models import UniDepthV2
    model = UniDepthV2.from_pretrained('lpiccinelli/unidepth-v2-vitl14').eval()
    rgb = torch.from_numpy(img_rgb).permute(2,0,1)
    K = torch.tensor([[f,0,cx],[0,f,cy],[0,0,1]], dtype=torch.float32)
    with torch.no_grad():
        pred = model.infer(rgb, K)
    return pred['depth'].squeeze().cpu().numpy().astype('float32')   # metric z, metres

def infer_metric3d_v2(img_rgb):
    import torch
    model = torch.hub.load('yvanyin/metric3d', 'metric3d_vit_small', pretrain=True).cuda().eval()
    # NOTE: Metric3D needs its canonical-camera resize + intrinsic scaling. TODO: port the
    # official demo transform (intrinsic = [f,f,cx,cy]); this stub will need that to be metric.
    raise NotImplementedError('port Metric3D canonical transform before use')

BACKENDS = {'depth_pro': infer_depth_pro, 'unidepth_v2': infer_unidepth_v2,
            'metric3d_v2': infer_metric3d_v2}

## 6. Eval loop — loc error at the standoff
Depth at a pixel → `point_at_depth` → world → error vs known centre. Scored two ways:
**GT pixel** (isolates depth accuracy) and **SAM pixel** (full detection+depth pipeline).

In [ ]:
def eval_depth(infer, use_sam=False, prompt='red cube'):
    loc, zerr, n = [], [], 0
    for fr in FRAMES:
        img = np.array(Image.open('data/'+fr['png']).convert('RGB'))
        P_gt = np.array(fr['gt_xyz'], float)
        if use_sam:
            c = sam3_center(img, prompt)
            if c is None: continue
            u, v = c
        else:
            u, v = fr['gt_uv']
        depth = infer(img)
        d = float(depth[int(round(v)), int(round(u))])
        P = CM.point_at_depth(u, v, f, cx, cy, cam_pos, R_cw, d)
        loc.append(np.linalg.norm(P - P_gt)*1000); zerr.append(abs(P[2]-P_gt[2])*1000); n += 1
    L = np.array(loc)
    return {'n': n, 'loc_mean_mm': float(L.mean()), 'loc_median_mm': float(np.median(L)),
            'loc_p95_mm': float(np.percentile(L,95)), 'z_median_mm': float(np.median(zerr))}

In [ ]:
# Run per installed backend (GT-pixel first to isolate depth quality):
results = {}
for name, infer in BACKENDS.items():
    try:
        results[name] = {'at_gt_px': eval_depth(infer, use_sam=False)}
        # results[name]['full_pipeline'] = eval_depth(infer, use_sam=True)  # needs SAM installed
    except Exception as e:
        results[name] = {'error': f'{type(e).__name__}: {str(e)[:80]}'}
for k, v in results.items(): print(f'{k:14s}', v)

## 7. Compare to the pipelines we already measured
Our earlier SO-101 numbers (same sim, same ~1.15 m side-cam standoff, ~2 mm grasp basin):

| pipeline | detector | depth source | loc error | notes |
|---|---|---|---|---|
| HSV + plane | red threshold | known plane z=0.015 (prior) | ~2.0 mm | colour + plane priors |
| SAM + plane | FastSAM (colour-free) | known plane z=0.015 (prior) | ~1.7 mm | plane prior |
| mono SAM+depth | FastSAM | Depth-Anything (affine) | ~12 mm | honest, no plane |
| 3-cam triangulation | FastSAM | ray geometry | **1.1 / z 0.6 mm** | honest, multi-view |

The rows in `results` are the **new single-camera, plane-free** contenders. The bar:
**loc median (ideally p95) ≤ ~2 mm at the GT pixel** ⇒ single-shot metric depth can carry the
cube; else it positions coarsely and the wrist-camera servo closes the last mm (container is fine
at cm regardless).

In [ ]:
REFERENCE = {
    'hsv_plane':        {'loc_median_mm': 2.0,  'depth': 'known plane (prior)'},
    'sam_plane':        {'loc_median_mm': 1.7,  'depth': 'known plane (prior)'},
    'mono_sam_depth':   {'loc_median_mm': 12.0, 'depth': 'Depth-Anything affine'},
    'triangulation_3cam':{'loc_median_mm': 1.1, 'depth': 'multi-view geometry'},
}
print('--- reference (measured earlier) ---')
for k,v in REFERENCE.items(): print(f'  {k:20s} loc_median {v["loc_median_mm"]:5.1f} mm  ({v["depth"]})')
print('--- new contenders (this run) ---')
for k,v in results.items(): print(f'  {k:20s} {v}')

## 8. Single-image visualization — the known centre vs each estimate

In [ ]:
import matplotlib.pyplot as plt
img = np.array(Image.open('data/'+hero['png']).convert('RGB'))
gu, gv = hero['gt_uv']
plt.figure(figsize=(7,5)); plt.imshow(img)
plt.scatter([gu],[gv], marker='+', s=200, c='lime', label='exact centre (GT)')
# overlay a SAM detection if available:
try:
    c = sam3_center(img)
    if c: plt.scatter([c[0]],[c[1]], marker='x', s=120, c='red', label='SAM 3')
except Exception as e:
    print('SAM not run:', type(e).__name__)
plt.legend(); plt.title(f"{hero['png']}  |  GT world {np.round(hero['gt_xyz'],3)}"); plt.axis('off'); plt.show()

## 9. Decision & how to iterate
- **Winner** = lowest `loc_median_mm` at the GT pixel that also survives the SAM-pixel (full) run.
- Register it back in the main repo as a `MetricDepthBackend` + `PERCEPT_DEPTH_MODEL`.
- **Broken cell?** Tell the author the cell + exact error; they push, you `!git pull` and re-run.
- Regenerate/extend the dataset from the main repo:
  `MUJOCO_GL=egl venv/bin/python experiments/export_depth_benchmark_data.py --nx 5 --ny 9 --raised`.

## 10. Push the run log to GitHub (`logs/` folder)
**One-time setup:** Colab left sidebar → 🔑 **Secrets** → **Add new secret** named `GH_TOKEN`,
value = a GitHub Personal Access Token with **write** access to this repo (Yunsmn account,
`repo` scope or fine-grained Contents: Read/write), and toggle **Notebook access ON**.

Then run the cell below at the end of every run. It copies `run_log.md` into `logs/run_<ts>.md`
and pushes it. The author does `git pull`, reads it, fixes the code; you push the fix and re-run.
The token stays in Colab Secrets — it never touches the notebook or the public repo, and any git
output that could echo it is redacted.

In [ ]:
# Push run_log.md -> logs/run_<timestamp>.md on GitHub. Token comes from Colab Secrets.
import subprocess, datetime, os, shutil
from google.colab import userdata

def _sh(args, secret=None):
    r = subprocess.run(args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret and out: out = out.replace(secret, '***')
    if out: print(out)
    return r.returncode

try:
    _TOKEN = userdata.get('GH_TOKEN')
except Exception as _e:
    _TOKEN = None; print('No GH_TOKEN secret:', _e)

if not _TOKEN:
    print('Set GH_TOKEN in the Colab 🔑 Secrets panel, then re-run. (Fallback: download_log())')
else:
    _REPO = 'Yunsmn/RoboticsPerceptionTest'
    try: _f.flush()
    except Exception: pass
    os.makedirs('logs', exist_ok=True)
    _stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    _dest = 'logs/run_%s.md' % _stamp
    shutil.copy('run_log.md', _dest)
    _url = 'https://%s@github.com/%s.git' % (_TOKEN, _REPO)
    _sh(['git','config','user.email','younesosf@gmail.com'])
    _sh(['git','config','user.name','Yunsmn'])
    _sh(['git','pull','--rebase','--autostash', _url, 'main'], secret=_TOKEN)
    _sh(['git','add', _dest])
    _sh(['git','commit','-m','log: run %s' % _stamp])
    _rc = _sh(['git','push', _url, 'HEAD:main'], secret=_TOKEN)
    print(('PUSHED ' if _rc == 0 else 'PUSH FAILED (rc=%d) ' % _rc) + _dest)
    print('-> tell the author: pull and read ' + _dest)